# production_summary

### Imports

In [0]:
# ============================================================================
# SMART MANUFACTURING INTELLIGENCE PLATFORM (SMIP)
# Gold Layer
#
# Notebook : 01_production_summary
# Layer    : Gold
#
# Description
# ----------------------------------------------------------------------------
# Creates the business production summary table from Silver facts and
# dimensions. This dataset is consumed directly by Power BI.
#
# Grain
# ----------------------------------------------------------------------------
# Production Date
# +
# Product
# +
# Planned Shift
#
# Author : Sumanth Vempalle
# ============================================================================

# ============================================================================
# Imports
# ============================================================================

from pyspark.sql.functions import (
    col,
    countDistinct,
    avg,
    sum,
    round,
    year,
    quarter,
    month,
    weekofyear,
    to_date,
    date_format
)

from framework.core.session import spark

from framework.core.configuration import (
    SILVER_LAYER,
    GOLD_LAYER
)

from framework.core.logger import (
    banner,
    info,
    success
)

from framework.io.delta import write_delta

In [0]:
# ============================================================================
# Start Notebook
# ============================================================================

banner("Gold Layer - Production Summary")

### Read Silver Tables

In [0]:
# ============================================================================
# Read Silver Tables
# ============================================================================

production = (

    spark.table(f"{SILVER_LAYER}.fact_production")

    .select(

        "production_key",

        "product_key",

        "serial_number",

        "execution_id",

        "work_order_id",

        "product_code",

        "product_name",

        "quantity",

        "planned_shift",

        "manufacturing_date"

    )

)

products = (

    spark.table(f"{SILVER_LAYER}.dim_products")

    .select(

        "product_key",

        "family",

        "rated_voltage_kv",

        "rated_current_a"

    )

)

info("Silver tables loaded successfully.")

### Join Product Information

In [0]:
# ============================================================================
# Join Product Dimension
# ============================================================================

df = (

    production.alias("p")

    .join(

        products.alias("d"),

        "product_key",

        "left"

    )

    .select(

        col("p.production_key"),

        col("p.product_key"),

        col("p.serial_number"),

        col("p.execution_id"),

        col("p.work_order_id"),

        col("p.product_code"),

        col("p.product_name"),

        col("p.quantity"),

        col("p.planned_shift"),

        col("p.manufacturing_date"),

        col("d.family"),

        col("d.rated_voltage_kv"),

        col("d.rated_current_a")

    )

)

### Create Date Attributes

In [0]:
# ============================================================================
# Calendar Attributes
# ============================================================================

df = (

    df

    .withColumn(

        "production_date",

        to_date("manufacturing_date")

    )

    .withColumn(

        "production_year",

        year("manufacturing_date")

    )

    .withColumn(

        "production_quarter",

        quarter("manufacturing_date")

    )

    .withColumn(

        "production_month",

        month("manufacturing_date")

    )

    .withColumn(

        "production_week",

        weekofyear("manufacturing_date")

    )

    .withColumn(

        "production_day",

        date_format(

            "manufacturing_date",

            "EEEE"

        )

    )

)

### Business Aggregation

In [0]:
# ============================================================================
# Production Summary
# ============================================================================

production_summary = (

    df

    .groupBy(

        "production_date",

        "production_year",

        "production_quarter",

        "production_month",

        "production_week",

        "production_day",

        "planned_shift",

        "product_key",

        "product_code",

        "product_name",

        "family",

        "rated_voltage_kv",

        "rated_current_a"

    )

    .agg(

        countDistinct(

            "serial_number"

        ).alias(

            "units_produced"

        ),

        countDistinct(

            "work_order_id"

        ).alias(

            "completed_work_orders"

        ),

        countDistinct(

            "execution_id"

        ).alias(

            "executions"

        ),

        round(

            avg("quantity"),

            2

        ).alias(

            "average_order_quantity"

        )

    )

)

### Write Gold

In [0]:
# ============================================================================
# Write Gold
# ============================================================================

write_delta(

    production_summary,

    f"{GOLD_LAYER}.production_summary"

)

success("gold.production_summary created successfully.")

### Validation

In [0]:
# ============================================================================
# Validation
# ============================================================================

display(production_summary)

display(

    spark.sql(f"""

    SELECT
        COUNT(*) AS rows
    FROM {GOLD_LAYER}.production_summary

    """)

)

display(

    spark.sql(f"""

    SELECT

        production_date,

        SUM(units_produced) AS production

    FROM {GOLD_LAYER}.production_summary

    GROUP BY production_date

    ORDER BY production_date

    """)

)

display(

    spark.sql(f"""

    SELECT

        planned_shift,

        SUM(units_produced) AS production

    FROM {GOLD_LAYER}.production_summary

    GROUP BY planned_shift

    ORDER BY production DESC

    """)

)

display(

    spark.sql(f"""

    SELECT

        product_name,

        SUM(units_produced) AS production

    FROM {GOLD_LAYER}.production_summary

    GROUP BY product_name

    ORDER BY production DESC

    """)

)
